In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('.').resolve()))

import utils.get_ahn_tiles as get_ahn_tiles
import utils.las_utils as las_utils

import nest_asyncio
nest_asyncio.apply()

import geopandas as gpd
from config import TILE_DIR, AHN_RAW_DIR, AHN_GRID_SHP, SETUP_TILECODES

In [ ]:
tilecodes = SETUP_TILECODES

data_dir      = TILE_DIR
ahn_grid_path = AHN_GRID_SHP
output_dir    = AHN_RAW_DIR

crs = "EPSG:28992"

In [ ]:
gdf_ahn = gpd.read_file(ahn_grid_path)
ahn_tiles = []

for tilecode in tilecodes:

    input_file = Path(data_dir) / f"{tilecode}.laz"
    poly = las_utils.build_convex_hull_polygon(input_file)
    centroid = poly.centroid

    gdf_point = gpd.GeoDataFrame(
        {"tilecode": [tilecode]},
        geometry=[centroid],
        crs=crs
    )

    joined = gpd.sjoin(gdf_point, gdf_ahn)
    ahn_tiles.extend(joined["GT_AHNSUB"].unique().tolist())

ahn_tiles = list(set(ahn_tiles))

In [ ]:
await get_ahn_tiles.download_all_tiles(
    ahn_tiles,
    output_dir,
    "https://geotiles.citg.tudelft.nl/AHN5_T/{code}.LAZ"
)

In [ ]:
!pip install -r requirements.txt